In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Retail Sales Dashboard") \
    .getOrCreate()

print(spark.version)

4.0.3


In [3]:
from google.colab import files
uploaded = files.upload()

Saving products.csv to products.csv
Saving sales.csv to sales.csv


In [4]:
sales_df = spark.read.csv(
    "sales.csv",
    header=True,
    inferSchema=True
)

products_df = spark.read.csv(
    "products.csv",
    header=True,
    inferSchema=True
)

In [5]:
sales_df.show()
products_df.show()

+-------+----------+--------+-----------+--------+----------+
|sale_id|product_id|store_id|employee_id|quantity| sale_date|
+-------+----------+--------+-----------+--------+----------+
|      1|      P101|    S101|        201|       2|2026-06-01|
|      2|      P102|    S102|        202|       3|2026-06-01|
|      3|      P103|    S103|        203|       5|2026-06-01|
|      4|      P104|    S104|        204|       4|2026-06-01|
|      5|      P105|    S105|        205|       2|2026-06-01|
|      6|      P106|    S101|        206|       3|2026-06-01|
|      7|      P107|    S102|        207|       8|2026-06-01|
|      8|      P108|    S103|        208|      10|2026-06-01|
|      9|      P109|    S104|        209|       2|2026-06-01|
|     10|      P110|    S105|        210|       4|2026-06-01|
|     11|      P101|    S101|        201|       1|2026-06-02|
|     12|      P102|    S102|        202|       2|2026-06-02|
|     13|      P103|    S103|        203|       6|2026-06-02|
|     14

In [6]:
sales_df.printSchema()
products_df.printSchema()

root
 |-- sale_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- sale_date: date (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- cost: double (nullable = true)



In [7]:
sales_df.printSchema()
products_df.printSchema()

root
 |-- sale_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- sale_date: date (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- cost: double (nullable = true)



In [8]:
sales_products_df = sales_df.join(
    products_df,
    "product_id"
)
sales_products_df.show()

+----------+-------+--------+-----------+--------+----------+------------+-----------+-------+-------+
|product_id|sale_id|store_id|employee_id|quantity| sale_date|product_name|   category|  price|   cost|
+----------+-------+--------+-----------+--------+----------+------------+-----------+-------+-------+
|      P101|      1|    S101|        201|       2|2026-06-01|      Laptop|Electronics|65000.0|55000.0|
|      P102|      2|    S102|        202|       3|2026-06-01|  Smartphone|Electronics|30000.0|25000.0|
|      P103|      3|    S103|        203|       5|2026-06-01|  Headphones|Electronics| 2500.0| 1800.0|
|      P104|      4|    S104|        204|       4|2026-06-01|  Smartwatch|Electronics| 5000.0| 3500.0|
|      P105|      5|    S105|        205|       2|2026-06-01|      Tablet|Electronics|20000.0|16000.0|
|      P106|      6|    S101|        206|       3|2026-06-01|     Printer|Accessories|12000.0| 9000.0|
|      P107|      7|    S102|        207|       8|2026-06-01|    Keyboard

In [9]:
from pyspark.sql.functions import col

sales_products_df = sales_products_df.withColumn(
    "revenue",
    col("quantity") * col("price")
)
sales_products_df.show()

+----------+-------+--------+-----------+--------+----------+------------+-----------+-------+-------+--------+
|product_id|sale_id|store_id|employee_id|quantity| sale_date|product_name|   category|  price|   cost| revenue|
+----------+-------+--------+-----------+--------+----------+------------+-----------+-------+-------+--------+
|      P101|      1|    S101|        201|       2|2026-06-01|      Laptop|Electronics|65000.0|55000.0|130000.0|
|      P102|      2|    S102|        202|       3|2026-06-01|  Smartphone|Electronics|30000.0|25000.0| 90000.0|
|      P103|      3|    S103|        203|       5|2026-06-01|  Headphones|Electronics| 2500.0| 1800.0| 12500.0|
|      P104|      4|    S104|        204|       4|2026-06-01|  Smartwatch|Electronics| 5000.0| 3500.0| 20000.0|
|      P105|      5|    S105|        205|       2|2026-06-01|      Tablet|Electronics|20000.0|16000.0| 40000.0|
|      P106|      6|    S101|        206|       3|2026-06-01|     Printer|Accessories|12000.0| 9000.0| 3

In [10]:
sales_products_df = sales_products_df.withColumn(
    "cost_amount",
    col("quantity") * col("cost")
)

In [11]:
sales_products_df = sales_products_df.withColumn(
    "profit",
    col("revenue") - col("cost_amount")
)

sales_products_df.show()

+----------+-------+--------+-----------+--------+----------+------------+-----------+-------+-------+--------+-----------+-------+
|product_id|sale_id|store_id|employee_id|quantity| sale_date|product_name|   category|  price|   cost| revenue|cost_amount| profit|
+----------+-------+--------+-----------+--------+----------+------------+-----------+-------+-------+--------+-----------+-------+
|      P101|      1|    S101|        201|       2|2026-06-01|      Laptop|Electronics|65000.0|55000.0|130000.0|   110000.0|20000.0|
|      P102|      2|    S102|        202|       3|2026-06-01|  Smartphone|Electronics|30000.0|25000.0| 90000.0|    75000.0|15000.0|
|      P103|      3|    S103|        203|       5|2026-06-01|  Headphones|Electronics| 2500.0| 1800.0| 12500.0|     9000.0| 3500.0|
|      P104|      4|    S104|        204|       4|2026-06-01|  Smartwatch|Electronics| 5000.0| 3500.0| 20000.0|    14000.0| 6000.0|
|      P105|      5|    S105|        205|       2|2026-06-01|      Tablet|El

In [12]:
underperforming_products = sales_products_df.filter(
    col("quantity") < 5
)
underperforming_products.show()

+----------+-------+--------+-----------+--------+----------+------------+-----------+-------+-------+--------+-----------+-------+
|product_id|sale_id|store_id|employee_id|quantity| sale_date|product_name|   category|  price|   cost| revenue|cost_amount| profit|
+----------+-------+--------+-----------+--------+----------+------------+-----------+-------+-------+--------+-----------+-------+
|      P101|      1|    S101|        201|       2|2026-06-01|      Laptop|Electronics|65000.0|55000.0|130000.0|   110000.0|20000.0|
|      P102|      2|    S102|        202|       3|2026-06-01|  Smartphone|Electronics|30000.0|25000.0| 90000.0|    75000.0|15000.0|
|      P104|      4|    S104|        204|       4|2026-06-01|  Smartwatch|Electronics| 5000.0| 3500.0| 20000.0|    14000.0| 6000.0|
|      P105|      5|    S105|        205|       2|2026-06-01|      Tablet|Electronics|20000.0|16000.0| 40000.0|    32000.0| 8000.0|
|      P106|      6|    S101|        206|       3|2026-06-01|     Printer|Ac

In [13]:
from pyspark.sql.functions import sum

product_summary = sales_products_df.groupBy(
    "product_name"
).agg(
    sum("quantity").alias("total_quantity"),
    sum("revenue").alias("total_revenue")
)
product_summary.show()

+------------+--------------+-------------+
|product_name|total_quantity|total_revenue|
+------------+--------------+-------------+
|  Smartwatch|             7|      35000.0|
|      Webcam|             9|      22500.0|
|      Laptop|             3|     195000.0|
|       Mouse|            19|      15200.0|
|      Tablet|             3|      60000.0|
|     Printer|             5|      60000.0|
|    Keyboard|            15|      22500.0|
|  Smartphone|             5|     150000.0|
|     Monitor|             3|      45000.0|
|  Headphones|            11|      27500.0|
+------------+--------------+-------------+



In [14]:
store_summary = sales_products_df.groupBy(
    "store_id"
).agg(
    sum("revenue").alias("total_revenue")
)

store_summary.show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|    S105|      82500.0|
|    S102|     172500.0|
|    S104|      80000.0|
|    S101|     255000.0|
|    S103|      42700.0|
+--------+-------------+



In [15]:
from pyspark.sql.functions import avg

average_store_revenue = sales_products_df.groupBy(
    "store_id"
).agg(
    avg("revenue").alias("average_revenue")
)

average_store_revenue.show()

+--------+---------------+
|store_id|average_revenue|
+--------+---------------+
|    S105|        20625.0|
|    S102|        43125.0|
|    S104|        20000.0|
|    S101|        63750.0|
|    S103|        10675.0|
+--------+---------------+



In [16]:
product_summary.orderBy(
    col("total_quantity").desc()
).show()

+------------+--------------+-------------+
|product_name|total_quantity|total_revenue|
+------------+--------------+-------------+
|       Mouse|            19|      15200.0|
|    Keyboard|            15|      22500.0|
|  Headphones|            11|      27500.0|
|      Webcam|             9|      22500.0|
|  Smartwatch|             7|      35000.0|
|     Printer|             5|      60000.0|
|  Smartphone|             5|     150000.0|
|      Laptop|             3|     195000.0|
|      Tablet|             3|      60000.0|
|     Monitor|             3|      45000.0|
+------------+--------------+-------------+



In [17]:
store_summary.write.mode("overwrite").csv("store_summary_output")
product_summary.write.mode("overwrite").csv("product_summary_output")

In [18]:
from pyspark.sql.functions import month

sales_products_df = sales_products_df.withColumn(
    "month",
    month("sale_date")
)

In [19]:
from pyspark.sql.functions import avg

monthly_revenue = sales_products_df.groupBy(
    "store_id",
    "month"
).agg(
    avg("revenue").alias("average_monthly_revenue")
)

monthly_revenue.show()

+--------+-----+-----------------------+
|store_id|month|average_monthly_revenue|
+--------+-----+-----------------------+
|    S104|    6|                20000.0|
|    S103|    6|                10675.0|
|    S102|    6|                43125.0|
|    S101|    6|                63750.0|
|    S105|    6|                20625.0|
+--------+-----+-----------------------+

